# 06 · ML anomaly detection: what the model sees

Phase 7, Step 7.6. The Isolation Forest of `configs/ml.yml` learnt what normal Silver trips look
like; this notebook shows what it learnt from, where its threshold sits, how many trips it flags
per month and why, and what it adds to the rules on the detection experiments.

**Before running:**
```
python src/ml/train.py
python src/gold/rebuild.py                    # scores every month (ml stage)
python src/evaluation/run_experiments.py      # for section 6 (optional)
python src/evaluation/evaluate.py
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config
from evaluation import evaluate as ev
from evaluation import fault_injector as fi
from ml import train as mt
from ml.features import load_ml_config
from silver.process_silver import current_silver

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
INK, BLUE, SURFACE = "#52514e", "#3987e5", "#fcfcfb"

## Settings

In [ ]:
config = load_ml_config()
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", load_datasets_config().paths["duckdb"]))
model = mt.load_model(config)
meta = model.metadata
con = db.connect(DATABASE, read_only=True)
print(f"model {meta['model_id']} (scikit-learn {meta['sklearn_version']})")
print(f"  trained on {', '.join(meta['train_months'])}: {meta['train_rows']:,} rows")
print(f"  threshold {meta['threshold']:.4f} set on {meta['calibrate_month']}: flags "
      f"{meta['calibrate_flag_rate']:.2%} (target {meta['target_flag_rate']:.2%})")

## 1 · What the model learnt from

Training median, quartiles and typical spread of every feature. Missing values (a division by
zero) are filled with the median; the spread is the unit of the explanations.

In [ ]:
features = pd.DataFrame({
    "sql": meta["features"],
    "median": meta["medians"],
    "q25": {f: q[0] for f, q in meta["quartiles"].items()},
    "q75": {f: q[1] for f, q in meta["quartiles"].items()},
    "spread": meta["scales"],
})
features.round(3)

## 2 · Where the threshold sits

Anomaly scores of a sample of the calibration month; the threshold leaves the target share of
them above it.

In [ ]:
silver = current_silver(con, config.dataset)
calibration = mt.sample_features(con, config, silver[config.calibrate_month][0],
                                 config.rows_per_month, config.seed)
scores = model.scores(calibration)
fig = go.Figure(go.Histogram(x=scores, nbinsx=60, marker_color=BLUE,
                             hovertemplate="score %{x}: %{y} rows<extra></extra>"))
fig.add_vline(x=meta["threshold"], line_dash="dash", line_color=INK,
              annotation_text=f"threshold {meta['threshold']:.3f}")
fig.update_layout(title=f"Anomaly scores in {config.calibrate_month}", xaxis_title="score",
                  yaxis_title="rows", plot_bgcolor=SURFACE, bargap=0.05)
fig.show()
print(f"{(scores > meta['threshold']).mean():.2%} of {len(scores):,} rows above the threshold")

## 3 · Flag rate per month

In [ ]:
summary = con.execute(
    "SELECT period, rows_scored, rows_flagged, flag_rate, warnings FROM ml_month_summary "
    "WHERE dataset = ? AND model_id = ? ORDER BY period", [config.dataset, meta["model_id"]],
).df()
incidents = con.execute(
    "SELECT incident_id, first_seen_period, last_seen_period, severity, status, latest_detail "
    "FROM incidents WHERE rule_id = 'ml_anomaly' ORDER BY incident_id"
).df()
summary.assign(flag_pct=(100 * summary["flag_rate"]).round(2)).drop(columns="flag_rate")

In [ ]:
fig = go.Figure(go.Bar(x=summary["period"], y=100 * summary["flag_rate"], marker_color=BLUE,
                       hovertemplate="%{x}: %{y:.2f}% flagged<extra></extra>"))
fig.add_hline(y=100 * meta["target_flag_rate"], line_dash="dash", line_color=INK,
              annotation_text="target")
fig.update_layout(title="Share of trips flagged per month", yaxis_title="% of Silver rows",
                  plot_bgcolor=SURFACE, bargap=0.25)
fig.show()
print(f"{len(incidents)} ML incident(s)")
incidents

## 4 · Why trips are flagged

The leading reason of every flag, and the most unusual trips with the values that made them
stand out.

In [ ]:
reasons = con.execute(
    "SELECT top_features[1] AS leading_feature, count(*) AS flags FROM ml_anomalies "
    "WHERE dataset = ? AND model_id = ? GROUP BY 1 ORDER BY 2 DESC",
    [config.dataset, meta["model_id"]],
).df()
reasons

In [ ]:
top = con.execute(
    "SELECT period, bronze_row_id, score, reasons FROM ml_anomalies WHERE dataset = ? "
    "AND model_id = ? ORDER BY score DESC LIMIT 10", [config.dataset, meta["model_id"]],
).df()
top.round({"score": 4})

## 5 · The model's known limit

An Isolation Forest isolates values by random splits within the training range: a value far
beyond the range scores about the same as one just beyond it. So it ranks unusual trips well,
but it cannot grade how extreme they are, and a month's oddities share the flag budget.

In [ ]:
example = calibration.head(1).copy()
column = "fare_per_mile"
rows = []
for factor in (1, 2, 5, 20, 100):
    trip = example.copy()
    trip[column] = meta["quartiles"][column][1] * factor
    rows.append({f"{column} (x q75)": factor, "score": float(model.scores(trip)[0])})
pd.DataFrame(rows).round(4)

## 6 · Rules only against rules + ML

From the detection experiments (Phase 6, `notebooks/05_detection_evaluation.ipynb`), when they
have been run with this setup.

In [ ]:
faults_path = os.environ.get("DATAGUARD_FAULTS_CONFIG")
faults = fi.load_faults_config(Path(faults_path) if faults_path else None)
try:
    results = ev.evaluate(faults)
    table = ev.comparison(results)
    control_rate, control_flags = ev.control_ml(faults)
except FileNotFoundError as exc:
    results, table, control_rate, control_flags = None, None, None, 0
    print(f"no experiments yet: {exc}")
table

In [ ]:
if table is not None:
    counts = pd.DataFrame({
        "rules only": table["rules_outcome"].value_counts(),
        "rules + ML": table["combined_outcome"].value_counts(),
    }).reindex(ev.OUTCOMES).fillna(0).astype(int)
    print(f"the model flags {control_rate:.2%} ({control_flags} rows) of the unbroken month")
    display(counts)

## 7 · Verdict

Copy the printed text back.

In [ ]:
verdict = {
    "model": meta["model_id"],
    "train_rows": meta["train_rows"],
    "threshold": round(meta["threshold"], 4),
    "calibrate_flag_rate": round(meta["calibrate_flag_rate"], 4),
    "flag_rate_by_month": {p: round(r, 4) for p, r in zip(summary["period"], summary["flag_rate"])},
    "months_above_twice_target": summary.loc[
        summary["flag_rate"] > 2 * meta["target_flag_rate"], "period"].tolist(),
    "ml_incidents": incidents["first_seen_period"].tolist(),
    "leading_reasons": reasons.head(3).set_index("leading_feature")["flags"].to_dict(),
    "warnings": sorted(set(summary["warnings"].dropna())),
}
if table is not None:
    verdict["comparison"] = {
        "rules_only_caught": int((table["rules_outcome"] == "caught").sum()),
        "rules_plus_ml_caught": int((table["combined_outcome"] == "caught").sum()),
        "ml_adds": table.loc[table["ml_adds"], "experiment_id"].tolist(),
        "control_flag_rate": None if control_rate is None else round(control_rate, 4),
    }
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
con.close()